# Diagnostic FP16 versus FP32
Gunakan kernel SongFormer Python 3.11 pada Mac, root Section-Annotations.
Paket FP16 lama dipakai ulang. FP32 diekspor ke folder baru. Semua validasi
menggunakan cache dan manual labels yang sama, tanpa test atau training ulang.
Diagnostic mode menyelesaikan laporan walau toleransi numerik gagal; ini bukan
izin integrasi/kompresi dan tidak melonggarkan pemeriksaan.


In [ ]:
import subprocess, sys, json
from pathlib import Path
from datetime import datetime
assert Path('finetuning/config.json').is_file()
FP16_PACKAGE = Path('outputs.local/coreml-fp16-20261114-180343/SongFormerFunctionHead.mlpackage')
assert FP16_PACKAGE.is_dir(), f'Periksa lokasi paket FP16: {FP16_PACKAGE}'
checkpoint = Path('outputs.local/finetuning/runs/worship-v3/best.pt')
comparison_dir = Path('outputs.local') / ('coreml-comparison-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
comparison_dir.mkdir(exist_ok=False)

def run_logged(command, log_path):
    with log_path.open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if code:
        raise RuntimeError(f'Proses gagal ({code}); lihat {log_path}')
print('Laporan:', comparison_dir)


Evaluasi ulang FP16 pada semua lagu validasi. Numerik yang gagal tetap ditandai, tetapi semua laporan diselesaikan.

In [ ]:
run_logged([sys.executable, '-u', '-m', 'deployment.verify_coreml',
    '--checkpoint', str(checkpoint), '--package', str(FP16_PACKAGE),
    '--candidate-name', 'fp16', '--diagnostic', '--limit', '0',
    '--output', str(comparison_dir / 'fp16-validation.json')], comparison_dir / 'fp16-validation.log')


Buat FP32 diagnostik. Paket FP16 dan resource SongMap asli tidak ditimpa.

In [ ]:
fp32_dir = comparison_dir / 'fp32-export'
run_logged([sys.executable, '-u', '-m', 'deployment.export_coreml',
    '--checkpoint', str(checkpoint), '--precision', 'fp32',
    '--output-dir', str(fp32_dir)], comparison_dir / 'fp32-export.log')


In [ ]:
run_logged([sys.executable, '-u', '-m', 'deployment.verify_coreml',
    '--checkpoint', str(checkpoint), '--package', str(fp32_dir / 'SongFormerFunctionHead.mlpackage'),
    '--candidate-name', 'fp32', '--diagnostic', '--limit', '0',
    '--output', str(comparison_dir / 'fp32-validation.json')], comparison_dir / 'fp32-validation.log')


In [ ]:
reports = {name: json.loads((comparison_dir / f'{name}-validation.json').read_text())
           for name in ('fp16', 'fp32')}
a, b = reports.values()
for key in ('plan_id', 'provenance_id', 'checkpoint_sha256', 'annotations_id', 'tracks'):
    assert a[key] == b[key], f'Perbandingan tidak setara: {key}'
assert a['full_validation'] and b['full_validation']
for name, r in reports.items():
    checks = r['chunk_checks']
    print(name.upper(), '| tracks:', r['tracks'], '| status:', r['status'])
    print('  PyTorch F1:', r['pytorch']['macro_f1_present'])
    print('  Core ML F1:', r['coreml']['macro_f1_present'], '| delta:', r['macro_f1_delta'])
    print('  Section agreement:', r['section_prediction_agreement'])
    print('  Max logit error:', max(c['max_abs_error'] for c in checks))
    print('  Max 8-class probability error:', max(c['allowed_probability_max_abs_error'] for c in checks))
    print('  Numerical checks failed:', sum(not c['passed'] for c in checks), '/', len(checks))
print('Belum otomatis disetujui untuk integrasi. Periksa per-class F1 dan confusion matrix dalam JSON.')
